In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder

# ── LOAD MODELS ──
xgb_regressor  = joblib.load('xgboost_regressor.pkl')
xgb_classifier = joblib.load('xgboost_classifier.pkl')
le_pollutant   = joblib.load('pollutant_encoder.pkl')
feature_cols   = joblib.load('feature_columns.pkl')

print("All models loaded!")

# ── ALERT SYSTEM FUNCTION ──
def get_aqi_alert(aqi):
    if aqi <= 50:
        return {
            'category': 'Good',
            'color': '🟢',
            'health_message': 'Air quality is satisfactory. Enjoy outdoor activities.',
            'advice': 'No precautions needed.'
        }
    elif aqi <= 100:
        return {
            'category': 'Satisfactory',
            'color': '🟡',
            'health_message': 'Air quality is acceptable for most people.',
            'advice': 'Unusually sensitive individuals should consider reducing prolonged outdoor activity.'
        }
    elif aqi <= 200:
        return {
            'category': 'Moderate',
            'color': '🟠',
            'health_message': 'Members of sensitive groups may experience health effects.',
            'advice': 'Wear a mask outdoors. Children and elderly should limit prolonged exertion.'
        }
    elif aqi <= 300:
        return {
            'category': 'Poor',
            'color': '🔴',
            'health_message': 'Everyone may begin to experience health effects.',
            'advice': 'Avoid outdoor activity. Keep windows closed. Use air purifier if available.'
        }
    elif aqi <= 400:
        return {
            'category': 'Very Poor',
            'color': '🟣',
            'health_message': 'Health warnings of emergency conditions for entire population.',
            'advice': 'Stay indoors. Wear N95 mask if going out. Avoid all physical outdoor activity.'
        }
    else:
        return {
            'category': 'Severe',
            'color': '⚫',
            'health_message': 'HEALTH EMERGENCY — Entire population is likely to be affected.',
            'advice': 'Do NOT go outside. Seal windows and doors. Seek medical attention if experiencing symptoms.'
        }

# ── PREDICTION FUNCTION ──
def predict_aqi(input_data):
    """
    input_data: dictionary with all feature values
    """
    # Convert to dataframe
    input_df = pd.DataFrame([input_data])

    # Ensure correct column order
    input_df = input_df[feature_cols]

    # Predict AQI
    predicted_aqi = xgb_regressor.predict(input_df)[0]
    predicted_aqi = max(0, min(500, predicted_aqi))  # clip to 0-500

    # Predict dominant pollutant
    predicted_class = xgb_classifier.predict(input_df)[0]
    predicted_pollutant = le_pollutant.inverse_transform([predicted_class])[0]

    # Get alert
    alert = get_aqi_alert(predicted_aqi)

    return predicted_aqi, predicted_pollutant, alert

# ── TEST WITH 3 SAMPLE SCENARIOS ──
print("\n" + "="*60)
print("MUMBAI AQI FORECASTING & ALERT SYSTEM — TEST SCENARIOS")
print("="*60)

scenarios = [
    {
        'name': 'Winter Morning (High Pollution Expected)',
        'data': {
            'year': 2024, 'day': 15, 'hour': 8, 'day_of_week': 1,
            'is_weekend': 0, 'season': 3,  # winter=3
            'temp_c': 18.0, 'humidity_percent': 75, 'dew_point_c': 13.5,
            'wind_speed_kmh': 5.0, 'wind_dir_deg': 45, 'wind_gusts_kmh': 8.0,
            'precipitation_mm': 0.0, 'is_raining': 0, 'heavy_rain': 0,
            'pressure_msl_hpa': 1015.0, 'cloud_cover_percent': 20,
            'pm2_5_ugm3': 120.0, 'pm10_ugm3': 180.0, 'co_ugm3': 1500,
            'no2_ugm3': 85.0, 'so2_ugm3': 60.0, 'o3_ugm3': 30.0, 'dust_ugm3': 5,
            'festival_period': 0, 'crop_burning_season': 1,
            'AQI_lag1': 280.0, 'AQI_lag2': 265.0, 'AQI_lag24': 290.0,
            'pm2_5_lag1': 115.0, 'pm2_5_lag24': 125.0,
            'o3_lag1': 28.0, 'o3_lag24': 32.0,
            'AQI_roll3': 275.0, 'AQI_roll6': 270.0, 'AQI_roll24': 260.0,
            'pm2_5_roll3': 112.0, 'pm2_5_roll24': 108.0, 'is_peak_hour': 1
        }
    },
    {
        'name': 'Monsoon Afternoon (Clean Air Expected)',
        'data': {
            'year': 2024, 'day': 20, 'hour': 14, 'day_of_week': 3,
            'is_weekend': 0, 'season': 0,  # monsoon=0
            'temp_c': 28.0, 'humidity_percent': 92, 'dew_point_c': 26.5,
            'wind_speed_kmh': 18.0, 'wind_dir_deg': 225, 'wind_gusts_kmh': 25.0,
            'precipitation_mm': 8.5, 'is_raining': 1, 'heavy_rain': 0,
            'pressure_msl_hpa': 1002.0, 'cloud_cover_percent': 95,
            'pm2_5_ugm3': 12.0, 'pm10_ugm3': 22.0, 'co_ugm3': 180,
            'no2_ugm3': 15.0, 'so2_ugm3': 10.0, 'o3_ugm3': 35.0, 'dust_ugm3': 1,
            'festival_period': 0, 'crop_burning_season': 0,
            'AQI_lag1': 42.0, 'AQI_lag2': 45.0, 'AQI_lag24': 38.0,
            'pm2_5_lag1': 11.0, 'pm2_5_lag24': 13.0,
            'o3_lag1': 33.0, 'o3_lag24': 37.0,
            'AQI_roll3': 43.0, 'AQI_roll6': 44.0, 'AQI_roll24': 41.0,
            'pm2_5_roll3': 12.0, 'pm2_5_roll24': 11.5, 'is_peak_hour': 0
        }
    },
    {
        'name': 'Diwali Night (Severe Pollution Expected)',
        'data': {
            'year': 2024, 'day': 1, 'hour': 21, 'day_of_week': 2,
            'is_weekend': 0, 'season': 2,  # post_monsoon=2
            'temp_c': 24.0, 'humidity_percent': 60, 'dew_point_c': 15.0,
            'wind_speed_kmh': 3.0, 'wind_dir_deg': 90, 'wind_gusts_kmh': 5.0,
            'precipitation_mm': 0.0, 'is_raining': 0, 'heavy_rain': 0,
            'pressure_msl_hpa': 1010.0, 'cloud_cover_percent': 10,
            'pm2_5_ugm3': 175.0, 'pm10_ugm3': 380.0, 'co_ugm3': 1900,
            'no2_ugm3': 120.0, 'so2_ugm3': 110.0, 'o3_ugm3': 20.0, 'dust_ugm3': 8,
            'festival_period': 1, 'crop_burning_season': 0,
            'AQI_lag1': 420.0, 'AQI_lag2': 390.0, 'AQI_lag24': 85.0,
            'pm2_5_lag1': 165.0, 'pm2_5_lag24': 28.0,
            'o3_lag1': 22.0, 'o3_lag24': 45.0,
            'AQI_roll3': 400.0, 'AQI_roll6': 350.0, 'AQI_roll24': 180.0,
            'pm2_5_roll3': 160.0, 'pm2_5_roll24': 85.0, 'is_peak_hour': 1
        }
    }
]

for scenario in scenarios:
    aqi, pollutant, alert = predict_aqi(scenario['data'], )
    print(f"\n📍 Scenario: {scenario['name']}")
    print(f"   Predicted AQI      : {aqi:.0f}")
    print(f"   Category           : {alert['color']} {alert['category']}")
    print(f"   Dominant Pollutant : {pollutant}")
    print(f"   Health Message     : {alert['health_message']}")
    print(f"   Advice             : {alert['advice']}")
    print("-"*60)

print("\nAlert System working!")

All models loaded!

MUMBAI AQI FORECASTING & ALERT SYSTEM — TEST SCENARIOS

📍 Scenario: Winter Morning (High Pollution Expected)
   Predicted AQI      : 343
   Category           : 🟣 Very Poor
   Dominant Pollutant : PM2.5
   Health Message     : Health warnings of emergency conditions for entire population.
   Advice             : Stay indoors. Wear N95 mask if going out. Avoid all physical outdoor activity.
------------------------------------------------------------

📍 Scenario: Monsoon Afternoon (Clean Air Expected)
   Predicted AQI      : 40
   Category           : 🟢 Good
   Dominant Pollutant : O3
   Health Message     : Air quality is satisfactory. Enjoy outdoor activities.
   Advice             : No precautions needed.
------------------------------------------------------------

📍 Scenario: Diwali Night (Severe Pollution Expected)
   Predicted AQI      : 333
   Category           : 🟣 Very Poor
   Dominant Pollutant : PM2.5
   Health Message     : Health warnings of emergency c